# Business Entity Resolution Challenge

**Goal:** Match noisy business records across 3 independent sources (S1, S2, S3) so that records referring to the same real-world business are linked.

- **Source 1** is the deduplicated reference source.
- For every S1 entity we must find zero, one, or many matching records from S2 and/or S3.
- Evaluation uses macro-averaged **F₀.₅** (precision-heavy).

This notebook implements a full pipeline:
1. Data loading & exploratory analysis
2. Text cleaning / normalization
3. Blocking (candidate generation)
4. Similarity features + supervised matching model
5. Validation with official F₀.₅ metric
6. Inference on the test set and generation of `matching_results.tsv` + `candidate_pairs.tsv`

## 0. Setup & Imports

In [1]:
import os
import re
import ast
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Optional but highly recommended for good fuzzy matching
try:
    from rapidfuzz import fuzz, process
    from rapidfuzz.distance import Levenshtein, JaroWinkler
    HAS_RAPIDFUZZ = True
except ImportError:
    print("rapidfuzz not installed – falling back to difflib (slower & weaker)")
    import difflib
    HAS_RAPIDFUZZ = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

d:\Programming\Experimental REPO\Amazon_ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## 1. Paths & Configuration

Adjust `DATA_ROOT` to point to the folder that contains `dataset/train/` and `dataset/test/`.

In [2]:
# ---------- CONFIG ----------
DATA_ROOT = Path("dataset")          # change if your data lives elsewhere
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Validation split size (fraction of Source-1 entities held out)
VAL_SIZE = 0.2

# Blocking parameters (tune these!)
BLOCK_MAX_CANDIDATES = 50          # hard cap per S1 entity after ranking
NAME_SIM_THRESHOLD   = 0.55        # keep candidates above this rough score

print(f"Train dir : {TRAIN_DIR.resolve()}")
print(f"Test  dir : {TEST_DIR.resolve()}")
print(f"Output    : {OUTPUT_DIR.resolve()}")

Train dir : D:\Programming\Experimental REPO\Amazon_ML\code\business_entity_resolution\src\dataset\train
Test  dir : D:\Programming\Experimental REPO\Amazon_ML\code\business_entity_resolution\src\dataset\test
Output    : D:\Programming\Experimental REPO\Amazon_ML\code\business_entity_resolution\src\output


## 2. Load Data

In [3]:
def load_source(path: Path) -> pd.DataFrame:
    """Load a source TSV with explicit tab separator."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    # Basic sanity
    expected = {"entity_id", "business_name", "business_address", "country"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    return df


def load_ground_truth(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    # Normalise empty lists
    df["matched_entity_ids"] = df["matched_entity_ids"].fillna("").astype(str)
    return df


# ---------- Training data ----------
train_s1 = load_source(TRAIN_DIR / "train_source1.tsv")
train_s2 = load_source(TRAIN_DIR / "train_source2.tsv")
train_s3 = load_source(TRAIN_DIR / "train_source3.tsv")
gt       = load_ground_truth(TRAIN_DIR / "train_ground_truth.tsv")

print("=== Training set sizes ===")
print(f"Source 1 : {len(train_s1):,}")
print(f"Source 2 : {len(train_s2):,}")
print(f"Source 3 : {len(train_s3):,}")
print(f"GT rows  : {len(gt):,}")
print()
print("Sample Source-1 records:")
display(train_s1.head(3))
print("\nSample ground-truth:")
display(gt.head(5))

=== Training set sizes ===
Source 1 : 2,206,821
Source 2 : 5,034,616
Source 3 : 5,285,603
GT rows  : 2,206,821

Sample Source-1 records:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US



Sample ground-truth:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


In [4]:
# ---------- Test data (if present) ----------
test_s1 = test_s2 = test_s3 = None
if (TEST_DIR / "test_source1.tsv").exists():
    test_s1 = load_source(TEST_DIR / "test_source1.tsv")
    test_s2 = load_source(TEST_DIR / "test_source2.tsv")
    test_s3 = load_source(TEST_DIR / "test_source3.tsv")
    print("=== Test set sizes ===")
    print(f"Source 1 : {len(test_s1):,}")
    print(f"Source 2 : {len(test_s2):,}")
    print(f"Source 3 : {len(test_s3):,}")
else:
    print("Test files not found – will only run validation pipeline.")

Test files not found – will only run validation pipeline.


## 3. Exploratory Data Analysis

In [5]:
def basic_stats(df: pd.DataFrame, name: str):
    print(f"\n===== {name} =====")
    print(f"Rows          : {len(df):,}")
    print(f"Unique IDs    : {df['entity_id'].nunique():,}")
    print(f"Countries     : {df['country'].value_counts().to_dict()}")
    print(f"Null/empty name    : {(df['business_name'].str.strip() == '').sum()}")
    print(f"Null/empty address : {(df['business_address'].str.strip() == '').sum()}")
    print(f"Avg name length    : {df['business_name'].str.len().mean():.1f}")
    print(f"Avg address length : {df['business_address'].str.len().mean():.1f}")

basic_stats(train_s1, "Train Source 1")
basic_stats(train_s2, "Train Source 2")
basic_stats(train_s3, "Train Source 3")


===== Train Source 1 =====
Rows          : 2,206,821
Unique IDs    : 2,206,821
Countries     : {'US': 1323633, 'India': 883188}
Null/empty name    : 0
Null/empty address : 0
Avg name length    : 24.0
Avg address length : 52.1

===== Train Source 2 =====
Rows          : 5,034,616
Unique IDs    : 5,034,616
Countries     : {'US': 3016817, 'India': 2017799}
Null/empty name    : 0
Null/empty address : 168967
Avg name length    : 25.1
Avg address length : 46.2

===== Train Source 3 =====
Rows          : 5,285,603
Unique IDs    : 5,285,603
Countries     : {'US': 3170056, 'India': 2115547}
Null/empty name    : 0
Null/empty address : 175916
Avg name length    : 25.2
Avg address length : 46.7


In [6]:
# Ground-truth statistics
gt["n_matches"] = gt["matched_entity_ids"].apply(
    lambda x: 0 if x.strip() == "" else len(x.split(","))
)
print("Matches per Source-1 entity (train):")
print(gt["n_matches"].describe())
print("\nDistribution of #matches:")
print(gt["n_matches"].value_counts().sort_index().head(15))
print(f"\nSingleton rate (no matches): {(gt['n_matches'] == 0).mean():.1%}")

Matches per Source-1 entity (train):
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: n_matches, dtype: float64

Distribution of #matches:
n_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Singleton rate (no matches): 5.6%


## 4. Text Cleaning & Normalization

Business names and addresses contain abbreviations, legal suffixes, punctuation differences, transliterations, etc. We apply a consistent normalisation so that string similarity works better.

In [7]:
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Constants
# ------------------------------------------------------------------
LEGAL_SUFFIXES = {
    "ltd", "limited", "pvt", "private", "pvt ltd", "private limited",
    "llc", "llp", "inc", "incorporated", "corp", "corporation",
    "co", "company", "plc", "sa", "sas", "sarl", "gmbh", "ag",
    "bv", "nv", "oy", "ab", "as", "spa", "srl", "kft", "zrt",
    "pty", "pty ltd", "sdn bhd", "bhd", "kk", "kabushiki kaisha",
}

# Longest-first so "private limited" is matched before "limited"
_SUFFIX_LIST = sorted(LEGAL_SUFFIXES, key=len, reverse=True)
_SUFFIX_REGEX = re.compile(
    r"(?:\s+(?:" + "|".join(re.escape(s) for s in _SUFFIX_LIST) + r"))+$",
    flags=re.IGNORECASE
)

ADDR_ABBREV = {
    r"\brd\b": "road", r"\bst\b": "street", r"\bave\b": "avenue",
    r"\bblvd\b": "boulevard", r"\bdr\b": "drive", r"\bln\b": "lane",
    r"\bct\b": "court", r"\bpl\b": "place", r"\bcir\b": "circle",
    r"\bhwy\b": "highway", r"\bpkwy\b": "parkway", r"\bsq\b": "square",
    r"\bapt\b": "apartment", r"\bste\b": "suite", r"\bfl\b": "floor",
    r"\bn\b": "north", r"\bs\b": "south", r"\be\b": "east", r"\bw\b": "west",
    r"\bne\b": "northeast", r"\bnw\b": "northwest",
    r"\bse\b": "southeast", r"\bsw\b": "southwest",
}

# Pre-compile all address patterns once
_ADDR_PATTERNS = [(re.compile(pat, flags=re.IGNORECASE), repl)
                  for pat, repl in ADDR_ABBREV.items()]

# ------------------------------------------------------------------
# Vectorized helpers (no .map, no Python loops over rows)
# ------------------------------------------------------------------
def _normalize_text_series(s: pd.Series) -> pd.Series:
    """Lower-case, strip punctuation, collapse whitespace – pure vectorized."""
    s = s.fillna("").astype(str).str.lower()
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def normalize_name(s: pd.Series) -> pd.Series:
    """Normalize business name + remove trailing legal suffixes."""
    name = _normalize_text_series(s)
    # Single vectorized regex pass removes one or more trailing suffixes
    name = name.str.replace(_SUFFIX_REGEX, "", regex=True).str.strip()
    return name

def normalize_address(s: pd.Series) -> pd.Series:
    """Normalize address and expand abbreviations."""
    addr = _normalize_text_series(s)
    for pat, repl in _ADDR_PATTERNS:
        addr = addr.str.replace(pat, repl, regex=True)
    return addr

def add_normalized_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add normalized columns. Fully vectorized – works great on Windows.
    """
    df = df.copy()
    df["name_norm"]   = normalize_name(df["business_name"])
    df["addr_norm"]   = normalize_address(df["business_address"])
    df["country_norm"] = df["country"].fillna("").astype(str).str.strip().str.lower()
    df["full_text"]   = (df["name_norm"] + " " + df["addr_norm"]).str.strip()
    return df

# ------------------------------------------------------------------
# Usage (identical to your original interface)
# ------------------------------------------------------------------
train_s1 = add_normalized_columns(train_s1)
train_s2 = add_normalized_columns(train_s2)
train_s3 = add_normalized_columns(train_s3)

print("Normalization examples:")
display(train_s1[["business_name", "name_norm", "business_address", "addr_norm"]].head(5))

Normalization examples:


,business_name,name_norm,business_address,addr_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",797 lake town block a kolkata howrah west bengal


## 5. Official F₀.₅ Evaluation Metric

Macro-averaged F₀.₅ computed **per Source-1 entity**, then averaged.

$$F_{0.5} = \frac{1.25 \cdot P \cdot R}{0.25 \cdot P + R}$$

In [8]:
def f05_score(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return (1.25 * precision * recall) / (0.25 * precision + recall)


def evaluate_matching(
    predictions: dict[str, set[str]],
    ground_truth: dict[str, set[str]],
    verbose: bool = True,
) -> float:
    """
    Compute macro-averaged F0.5.

    Parameters
    ----------
    predictions : dict  source1_id → set of predicted S2/S3 ids
    ground_truth: dict  source1_id → set of true S2/S3 ids
    """
    scores = []
    for sid in ground_truth:
        true = ground_truth[sid]
        pred = predictions.get(sid, set())

        if len(true) == 0 and len(pred) == 0:
            scores.append(1.0)          # correct singleton
            continue
        if len(true) == 0 and len(pred) > 0:
            scores.append(0.0)          # false merges on singleton
            continue
        if len(pred) == 0:
            scores.append(0.0)          # missed everything
            continue

        tp = len(true & pred)
        precision = tp / len(pred)
        recall    = tp / len(true)
        scores.append(f05_score(precision, recall))

    macro = float(np.mean(scores))
    if verbose:
        print(f"Macro F0.5 = {macro:.5f}  (over {len(scores)} Source-1 entities)")
    return macro


def gt_to_dict(gt_df: pd.DataFrame) -> dict[str, set[str]]:
    d = {}
    for _, row in gt_df.iterrows():
        ids = set()
        if row["matched_entity_ids"].strip():
            ids = set(x.strip() for x in row["matched_entity_ids"].split(",") if x.strip())
        d[row["source1_entity_id"]] = ids
    return d

## 6. Train / Validation Split

We hold out a fraction of Source-1 entities (and their true matches) so we can tune blocking & the matcher without touching the official test set.

In [9]:
all_s1_ids = train_s1["entity_id"].tolist()
train_ids, val_ids = train_test_split(
    all_s1_ids, test_size=VAL_SIZE, random_state=RANDOM_STATE
)

train_s1_split = train_s1[train_s1["entity_id"].isin(train_ids)].reset_index(drop=True)
val_s1_split   = train_s1[train_s1["entity_id"].isin(val_ids)].reset_index(drop=True)

gt_dict = gt_to_dict(gt)
train_gt = {k: v for k, v in gt_dict.items() if k in set(train_ids)}
val_gt   = {k: v for k, v in gt_dict.items() if k in set(val_ids)}

print(f"Train S1 entities : {len(train_s1_split):,}")
print(f"Val   S1 entities : {len(val_s1_split):,}")
print(f"Train GT entries  : {len(train_gt):,}")
print(f"Val   GT entries  : {len(val_gt):,}")

KeyboardInterrupt: 

## 7. Blocking / Candidate Generation

We never compare every S1 record to every S2/S3 record (that would be too slow).
Instead we generate a small set of plausible candidates per S1 entity using several cheap keys:

1. **Exact country** (hard filter – records from different countries almost never match).
2. **Name token overlap** (Jaccard on name tokens).
3. **First-k characters of normalised name** (sorted-neighbourhood style).
4. **Address token overlap** (secondary signal).

Candidates are ranked by a cheap similarity score and truncated to `BLOCK_MAX_CANDIDATES`.

In [ ]:
def token_set(text: str) -> set[str]:
    return set(text.split()) if text else set()


def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)


def cheap_name_sim(n1: str, n2: str) -> float:
    """Fast approximate name similarity (0-1)."""
    if not n1 or not n2:
        return 0.0
    if HAS_RAPIDFUZZ:
        # token_sort_ratio handles word-order differences well
        return fuzz.token_sort_ratio(n1, n2) / 100.0
    else:
        return difflib.SequenceMatcher(None, n1, n2).ratio()


def build_inverted_index(df: pd.DataFrame, col: str = "name_norm", ngram: int = 3):
    """Simple character n-gram inverted index for blocking."""
    index = defaultdict(set)
    for idx, text in enumerate(df[col]):
        text = text.replace(" ", "")
        for i in range(max(1, len(text) - ngram + 1)):
            gram = text[i : i + ngram]
            index[gram].add(idx)
    return index


def generate_candidates(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    max_candidates: int = BLOCK_MAX_CANDIDATES,
    name_threshold: float = NAME_SIM_THRESHOLD,
) -> dict[str, list[str]]:
    """
    For every S1 entity return a ranked list of candidate entity_ids from S2 ∪ S3.

    Returns
    -------
    dict  source1_id → list of candidate ids (already sorted by descending score)
    """
    # Pre-index S2 and S3 by country for a cheap hard filter
    s2_by_country = {c: g for c, g in s2_df.groupby("country_norm")}
    s3_by_country = {c: g for c, g in s3_df.groupby("country_norm")}

    # Character n-gram indexes (helps with typos)
    s2_name_idx = build_inverted_index(s2_df, "name_norm", ngram=3)
    s3_name_idx = build_inverted_index(s3_df, "name_norm", ngram=3)

    candidates = {}

    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc="Blocking"):
        sid = row["entity_id"]
        name = row["name_norm"]
        addr = row["addr_norm"]
        country = row["country_norm"]

        cand_scores = {}  # entity_id → score

        # ----- candidates from same country -----
        for src_df, name_idx in [(s2_by_country.get(country, pd.DataFrame()), s2_name_idx),
                                 (s3_by_country.get(country, pd.DataFrame()), s3_name_idx)]:
            if src_df.empty:
                continue

            # 1) n-gram retrieval
            name_clean = name.replace(" ", "")
            retrieved = set()
            for i in range(max(1, len(name_clean) - 2)):
                gram = name_clean[i : i + 3]
                retrieved |= name_idx.get(gram, set())

            # Limit retrieval size for speed
            if len(retrieved) > 300:
                retrieved = set(list(retrieved)[:300])

            # Score the retrieved rows + a few random same-country rows as fallback
            pool_idx = list(retrieved)
            if len(pool_idx) < 30:
                # add some same-country rows so we never return empty when data exists
                extra = src_df.sample(min(30, len(src_df)), random_state=hash(sid) % 2**32).index.tolist()
                pool_idx = list(set(pool_idx) | set(extra))

            for idx in pool_idx:
                if idx not in src_df.index:
                    continue
                other = src_df.loc[idx]
                sim = cheap_name_sim(name, other["name_norm"])
                # slight boost if address tokens overlap
                addr_boost = 0.1 * jaccard(token_set(addr), token_set(other["addr_norm"]))
                score = sim + addr_boost
                if score >= name_threshold * 0.7:  # soft threshold
                    cand_scores[other["entity_id"]] = max(cand_scores.get(other["entity_id"], 0), score)

        # Rank & truncate
        ranked = sorted(cand_scores.items(), key=lambda x: -x[1])[:max_candidates]
        candidates[sid] = [eid for eid, _ in ranked]

    return candidates

In [ ]:
# Generate candidates on the validation split (for later scoring)
print("Generating candidates for validation split …")
val_candidates = generate_candidates(val_s1_split, train_s2, train_s3)

# Quick look at candidate-set size
cand_sizes = [len(v) for v in val_candidates.values()]
print(f"Avg candidates per S1 entity : {np.mean(cand_sizes):.1f}")
print(f"Median                       : {np.median(cand_sizes):.0f}")
print(f"Max                          : {np.max(cand_sizes)}")

### 7.1 Blocking Quality (Recall Ceiling)

If a true match never appears in the candidate list, the downstream model can never recover it. We therefore measure **blocking recall** on the validation set.

In [ ]:
def blocking_recall(candidates: dict[str, list[str]], gt: dict[str, set[str]]) -> float:
    hits, total = 0, 0
    for sid, true_ids in gt.items():
        if not true_ids:
            continue
        total += len(true_ids)
        cand_set = set(candidates.get(sid, []))
        hits += len(true_ids & cand_set)
    return hits / total if total else 1.0

br = blocking_recall(val_candidates, val_gt)
print(f"Blocking recall (validation) = {br:.3%}")
print("(Aim for ≥ 90-95 %; if lower, loosen thresholds or add more blocking keys.)")

## 8. Pairwise Feature Engineering

For every (S1, candidate) pair we compute a vector of similarity features that a classifier can use.

In [ ]:
def pairwise_features(row1: pd.Series, row2: pd.Series) -> dict:
    """Compute a rich set of similarity features between two records."""
    n1, n2 = row1["name_norm"], row2["name_norm"]
    a1, a2 = row1["addr_norm"], row2["addr_norm"]

    feats = {}

    # ---- Name features ----
    if HAS_RAPIDFUZZ:
        feats["name_ratio"]       = fuzz.ratio(n1, n2) / 100.0
        feats["name_partial"]    = fuzz.partial_ratio(n1, n2) / 100.0
        feats["name_token_sort"] = fuzz.token_sort_ratio(n1, n2) / 100.0
        feats["name_token_set"]  = fuzz.token_set_ratio(n1, n2) / 100.0
        feats["name_jw"]         = JaroWinkler.normalized_similarity(n1, n2)
        feats["name_lev"]        = Levenshtein.normalized_similarity(n1, n2)
    else:
        feats["name_ratio"] = difflib.SequenceMatcher(None, n1, n2).ratio()
        feats["name_partial"] = feats["name_token_sort"] = feats["name_token_set"] = feats["name_ratio"]
        feats["name_jw"] = feats["name_lev"] = feats["name_ratio"]

    # token Jaccard
    t1, t2 = token_set(n1), token_set(n2)
    feats["name_jaccard"] = jaccard(t1, t2)
    feats["name_len_ratio"] = min(len(n1), len(n2)) / max(len(n1), len(n2), 1)

    # ---- Address features ----
    if HAS_RAPIDFUZZ:
        feats["addr_ratio"]       = fuzz.ratio(a1, a2) / 100.0
        feats["addr_token_sort"]  = fuzz.token_sort_ratio(a1, a2) / 100.0
        feats["addr_token_set"]   = fuzz.token_set_ratio(a1, a2) / 100.0
    else:
        feats["addr_ratio"] = difflib.SequenceMatcher(None, a1, a2).ratio()
        feats["addr_token_sort"] = feats["addr_token_set"] = feats["addr_ratio"]

    ta1, ta2 = token_set(a1), token_set(a2)
    feats["addr_jaccard"] = jaccard(ta1, ta2)

    # ---- Country (should almost always match after blocking) ----
    feats["same_country"] = 1.0 if row1["country_norm"] == row2["country_norm"] else 0.0

    # ---- Combined signal ----
    feats["name_addr_avg"] = 0.6 * feats["name_token_sort"] + 0.4 * feats["addr_token_sort"]

    return feats


FEATURE_NAMES = [
    "name_ratio", "name_partial", "name_token_sort", "name_token_set",
    "name_jw", "name_lev", "name_jaccard", "name_len_ratio",
    "addr_ratio", "addr_token_sort", "addr_token_set", "addr_jaccard",
    "same_country", "name_addr_avg",
]

## 9. Build Training Pairs for the Classifier

We turn the ground-truth into labelled (S1, S2/S3) pairs and also sample hard negatives from the candidate lists.

In [ ]:
def build_labelled_pairs(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    gt: dict[str, set[str]],
    candidates: dict[str, list[str]] | None = None,
    neg_ratio: float = 3.0,
) -> pd.DataFrame:
    """
    Create a DataFrame of labelled pairs with features.

    Positive pairs come from ground truth.
    Negative pairs are sampled from the candidate list (or random same-country records).
    """
    # Lookup tables
    s2_lookup = s2_df.set_index("entity_id")
    s3_lookup = s3_df.set_index("entity_id")
    s1_lookup = s1_df.set_index("entity_id")

    rows = []

    for sid, true_ids in tqdm(gt.items(), desc="Building pairs"):
        if sid not in s1_lookup.index:
            continue
        r1 = s1_lookup.loc[sid]

        # ---- positives ----
        for mid in true_ids:
            if mid in s2_lookup.index:
                r2 = s2_lookup.loc[mid]
            elif mid in s3_lookup.index:
                r2 = s3_lookup.loc[mid]
            else:
                continue
            feats = pairwise_features(r1, r2)
            feats["source1_id"] = sid
            feats["candidate_id"] = mid
            feats["label"] = 1
            rows.append(feats)

        # ---- negatives ----
        cand_pool = candidates.get(sid, []) if candidates else []
        # remove true matches from the negative pool
        neg_pool = [c for c in cand_pool if c not in true_ids]
        n_neg = min(len(neg_pool), max(1, int(len(true_ids) * neg_ratio)) if true_ids else 5)
        if n_neg == 0 and not true_ids:
            # pure singleton – sample a few random same-country negatives
            country = r1["country_norm"]
            pool = pd.concat([
                s2_df[s2_df["country_norm"] == country],
                s3_df[s3_df["country_norm"] == country],
            ])
            if len(pool) > 0:
                neg_pool = pool.sample(min(5, len(pool)), random_state=hash(sid) % 2**32)["entity_id"].tolist()
                n_neg = len(neg_pool)

        for mid in neg_pool[:n_neg]:
            if mid in s2_lookup.index:
                r2 = s2_lookup.loc[mid]
            elif mid in s3_lookup.index:
                r2 = s3_lookup.loc[mid]
            else:
                continue
            feats = pairwise_features(r1, r2)
            feats["source1_id"] = sid
            feats["candidate_id"] = mid
            feats["label"] = 0
            rows.append(feats)

    return pd.DataFrame(rows)


# First generate candidates on the training split as well (needed for hard negatives)
print("Generating candidates for training split …")
train_candidates = generate_candidates(train_s1_split, train_s2, train_s3)

print("Building labelled training pairs …")
train_pairs = build_labelled_pairs(
    train_s1_split, train_s2, train_s3, train_gt, train_candidates, neg_ratio=4.0
)
print(f"Training pairs : {len(train_pairs):,}  (pos={train_pairs['label'].sum():,})")
display(train_pairs.head())

## 10. Train the Matching Classifier

In [ ]:
X_train = train_pairs[FEATURE_NAMES].fillna(0.0)
y_train = train_pairs["label"]

# Gradient Boosting works very well on this kind of tabular similarity features
clf = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    random_state=RANDOM_STATE,
)
clf.fit(X_train, y_train)

print("Feature importances:")
imp = pd.Series(clf.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
print(imp.round(3))

## 11. Inference Helper & Threshold Tuning

In [ ]:
def predict_matches(
    s1_df: pd.DataFrame,
    s2_df: pd.DataFrame,
    s3_df: pd.DataFrame,
    candidates: dict[str, list[str]],
    model,
    threshold: float = 0.5,
) -> dict[str, set[str]]:
    """
    Score every candidate pair and keep those above `threshold`.
    """
    s2_lookup = s2_df.set_index("entity_id")
    s3_lookup = s3_df.set_index("entity_id")
    s1_lookup = s1_df.set_index("entity_id")

    predictions = {}

    for sid, cand_ids in tqdm(candidates.items(), desc="Scoring"):
        if sid not in s1_lookup.index:
            predictions[sid] = set()
            continue
        r1 = s1_lookup.loc[sid]
        kept = set()

        if not cand_ids:
            predictions[sid] = set()
            continue

        feat_rows = []
        valid_cands = []
        for cid in cand_ids:
            if cid in s2_lookup.index:
                r2 = s2_lookup.loc[cid]
            elif cid in s3_lookup.index:
                r2 = s3_lookup.loc[cid]
            else:
                continue
            feat_rows.append(pairwise_features(r1, r2))
            valid_cands.append(cid)

        if not feat_rows:
            predictions[sid] = set()
            continue

        X = pd.DataFrame(feat_rows)[FEATURE_NAMES].fillna(0.0)
        proba = model.predict_proba(X)[:, 1]

        for cid, p in zip(valid_cands, proba):
            if p >= threshold:
                kept.add(cid)

        predictions[sid] = kept

    return predictions

In [ ]:
# Tune decision threshold on the validation set
print("Scoring validation candidates …")
val_pred_raw = predict_matches(
    val_s1_split, train_s2, train_s3, val_candidates, clf, threshold=0.0  # keep all scores
)

# Because predict_matches with threshold=0 returns everything,
# we re-score properly to get probabilities for threshold search.
# (For speed we already have the candidate lists; a lightweight re-implementation follows.)

def collect_val_scores():
    s2_lookup = train_s2.set_index("entity_id")
    s3_lookup = train_s3.set_index("entity_id")
    s1_lookup = val_s1_split.set_index("entity_id")
    scores = {}  # sid → list of (cid, proba)
    for sid, cands in tqdm(val_candidates.items(), desc="Collect scores"):
        r1 = s1_lookup.loc[sid]
        pair_scores = []
        for cid in cands:
            if cid in s2_lookup.index:
                r2 = s2_lookup.loc[cid]
            elif cid in s3_lookup.index:
                r2 = s3_lookup.loc[cid]
            else:
                continue
            feats = pairwise_features(r1, r2)
            X = pd.DataFrame([feats])[FEATURE_NAMES].fillna(0.0)
            p = clf.predict_proba(X)[0, 1]
            pair_scores.append((cid, p))
        scores[sid] = pair_scores
    return scores

val_scores = collect_val_scores()

In [ ]:
def predictions_from_scores(scores: dict, threshold: float) -> dict[str, set[str]]:
    return {
        sid: {cid for cid, p in pairs if p >= threshold}
        for sid, pairs in scores.items()
    }


best_thr, best_f05 = 0.5, -1.0
print("Threshold search:")
for thr in np.arange(0.30, 0.85, 0.05):
    preds = predictions_from_scores(val_scores, thr)
    # Ensure every validation S1 id is present
    for sid in val_gt:
        preds.setdefault(sid, set())
    f05 = evaluate_matching(preds, val_gt, verbose=False)
    print(f"  thr={thr:.2f}  →  F0.5={f05:.5f}")
    if f05 > best_f05:
        best_f05 = f05
        best_thr = thr

print(f"\nBest threshold = {best_thr:.2f}  (F0.5 = {best_f05:.5f})")

In [ ]:
# Final validation score with the chosen threshold
val_predictions = predictions_from_scores(val_scores, best_thr)
for sid in val_gt:
    val_predictions.setdefault(sid, set())

print("\n=== Validation Performance ===")
evaluate_matching(val_predictions, val_gt)

## 12. Full-Pipeline Inference on Test Set

If test files are present we run the complete pipeline and write the two required submission files.

In [ ]:
if test_s1 is not None:
    print("Normalising test data …")
    test_s1 = add_normalized_columns(test_s1)
    test_s2 = add_normalized_columns(test_s2)
    test_s3 = add_normalized_columns(test_s3)

    print("Generating test candidates (this may take a few minutes) …")
    test_candidates = generate_candidates(test_s1, test_s2, test_s3)

    print("Scoring test pairs …")
    test_predictions = predict_matches(
        test_s1, test_s2, test_s3, test_candidates, clf, threshold=best_thr
    )

    # Guarantee every Source-1 test entity appears
    for sid in test_s1["entity_id"]:
        test_predictions.setdefault(sid, set())
        test_candidates.setdefault(sid, [])

    print("Writing submission files …")

    # --- matching_results.tsv ---
    match_rows = []
    for sid in test_s1["entity_id"]:
        matched = sorted(test_predictions[sid])          # deterministic order
        match_rows.append({
            "source1_entity_id": sid,
            "matched_entity_ids": ",".join(matched),
        })
    match_df = pd.DataFrame(match_rows)
    match_path = OUTPUT_DIR / "matching_results.tsv"
    match_df.to_csv(match_path, sep="\t", index=False)
    print(f"Wrote {match_path}  ({len(match_df):,} rows)")

    # --- candidate_pairs.tsv ---
    cand_rows = []
    for sid in test_s1["entity_id"]:
        cands = test_candidates[sid]
        # de-duplicate while preserving order
        seen = set()
        unique = []
        for c in cands:
            if c not in seen:
                seen.add(c)
                unique.append(c)
        cand_rows.append({
            "source1_entity_id": sid,
            "candidate_entity_ids": ",".join(unique),
        })
    cand_df = pd.DataFrame(cand_rows)
    cand_path = OUTPUT_DIR / "candidate_pairs.tsv"
    cand_df.to_csv(cand_path, sep="\t", index=False)
    print(f"Wrote {cand_path}  ({len(cand_df):,} rows)")

    print("\nDone!  You can now run the official validator:")
    print("python3 utils/validate_submission.py \\")
    print("    --matching output/matching_results.tsv \\")
    print("    --candidate output/candidate_pairs.tsv \\")
    print("    --test-dir dataset/test")
else:
    print("No test data found – skipping final inference.")